# Playstyle analysis (EPL 2003/04 vs 2015/16)


Compare how Premier League playstyle changed between **2003/04** and **2015/16**.

**Headline questions:**
1. How many **passes per match**?
2. How **long are possessions** (events per possession, passes per possession, duration in seconds)?

Additional themes: pressing, direct play, physicality, chance quality.

**Note:** 2003/04 has only **38 matches** vs **380** in 2015/16 — use boxplots and treat means as illustrative.

## Per season analysis


In [12]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
import dill
import duckdb

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, RAW_DIR
from src.metrics import (
    ERA_2004,
    ERA_2016,
    EVENT_COLS,
    build_match_team,
    build_season_metrics
)
from src.progression import (
    _possession_stats_base,
    _possession_stats_progression,
    _possession_stats_time,
    _possession_directness_ratio,
    _possession_movement_events,
    add_possession_opponent,
    _possession_locations,
    _possession_stats_outcome,
    _possession_stats_pressure
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
matches_2004 = pd.read_csv(PROCESSED_DIR / "matches_2004.csv").assign(season=ERA_2004)
matches_2016 = pd.read_csv(PROCESSED_DIR / "matches_2016.csv").assign(season=ERA_2016)
matches_df = pd.concat([matches_2004, matches_2016], ignore_index=True)

events_2004_path = PROCESSED_DIR / "events_2004.parquet"
events_2016_path = PROCESSED_DIR / "events_2016.parquet"
events_df = pd.concat(
    [
        pd.read_parquet(events_2004_path, columns=EVENT_COLS),
        pd.read_parquet(events_2016_path, columns=EVENT_COLS),
    ],
    ignore_index=True,
)

In [14]:
match_meta = matches_df[["match_id", "season", "match_kickoff"]].drop_duplicates()
analysis_events = events_df.merge(match_meta, on="match_id", how="inner")

matches_by_season = match_meta.groupby("season").agg(matches=("match_id", "nunique"))
events_by_season = analysis_events.groupby("season").size().rename("events")

coverage = matches_by_season.join(events_by_season, how="left").fillna(0)
coverage["events"] = coverage["events"].astype(int)

print(f"Total matches in catalog: {match_meta['match_id'].nunique()}")
print(f"Total events captured: {len(analysis_events):,}\n")
print("Matches vs events captured by season:")
print(coverage)

if ERA_2016 in coverage.index:
    est_1516 = coverage.loc["2015/2016", "matches"] * 3404
    actual_1516 = coverage.loc["2015/2016", "events"]
    print(f"\n2015/16 expected ~{est_1516:,.0f} events | actual {actual_1516:,}")
if ERA_2004 in coverage.index:
    est_0304 = coverage.loc["2003/2004", "matches"] * 3404
    actual_0304 = coverage.loc["2003/2004", "events"]
    print(f"2003/04 expected ~{est_0304:,.0f} events | actual {actual_0304:,}")


Total matches in catalog: 418
Total events captured: 1,443,174

Matches vs events captured by season:
           matches   events
season                     
2003/2004       38   129401
2015/2016      380  1313773

2015/16 expected ~1,293,520 events | actual 1,313,773
2003/04 expected ~129,352 events | actual 129,401


In [15]:
match_team = build_match_team(analysis_events)
season_metrics = build_season_metrics(match_team)

print(season_metrics.head())


              match_id    events   passes  pass_completion  avg_pass_length  \
season                                                                        
2003/2004  3749347.947  1702.645  475.868            0.732           23.193   
2015/2016  3754161.500  1728.649  485.025            0.754           22.288   

           long_pass_pct  crosses  through_balls  pressure_on_pass_pct  \
season                                                                   
2003/2004          0.240   11.026          2.289                 0.144   
2015/2016          0.224   12.559          2.133                 0.156   

           counterpresses  ...  offensive_recoveries   shots     xg  \
season                     ...                                        
2003/2004          55.276  ...                 0.132  12.224  1.207   
2015/2016          55.689  ...                 0.184  13.037  1.278   

           regular_play_pct  passes_into_final_third  possession_count  \
season                     

# Extracting one match

Since the definitive teams during the 2003/2004 era were Arsene Wenger's Arsenal and Jose Mourinho's Chelsea, I plan to analyze either team's first match and compare their progression structure, playstyle, tempo etc. to their other matches over the season to see how it has progressed overall. I want to see how the team's tactical identity develops/shapes throughout the season. This would in turn tell us more about the playstyle that defined the "Barclay's" era.

I chose to analyze Arsenal's 2003/2004 season as they went unbeaten that season and went on to become champions. Arsenal's opening match of the season was vs. Everton with `match_id 3749493`. 

In [16]:
match_df = duckdb.sql("SELECT * FROM 'data/processed/events_2004.parquet' WHERE match_id = 3749493").df()
print(match_df.head().columns)
print(duckdb.sql("SELECT COUNT(*) FROM 'data/processed/events_2004.parquet' WHERE match_id = 3749493").df())

Index(['ball_receipt_outcome', 'ball_recovery_recovery_failure',
       'block_deflection', 'block_offensive', 'carry_end_location',
       'clearance_aerial_won', 'clearance_body_part', 'clearance_head',
       'clearance_left_foot', 'clearance_right_foot',
       ...
       '50_50', 'goalkeeper_success_in_play', 'injury_stoppage_in_chain',
       'clearance_other', 'goalkeeper_lost_out',
       'goalkeeper_shot_saved_off_target', 'shot_saved_off_target',
       'half_start_late_video_start', 'goalkeeper_shot_saved_to_post',
       'shot_saved_to_post'],
      dtype='object', length=124)
   count_star()
0          3057


## Possession analysis

We want to see whether the possession was structured vs. chaotic using metrics, such as `duration`, `event_count`, `pass_count`. This helps us understand:

1. How long can the team sustain control?
2. Are possession deliberate or rushed?
3. Does the team recycle possession or attack immediately?

This should provide insight to whether or not the early 2000s premier league fit the stereotype that it had shorter possession, fewer actions, more transitions and quicker vertical attacks. Some other questions to consider:

- How efficiently does the team move upfield?
- Is progression gradual or explosive?
- Does buildup happen through controlled advancement?
- Can the team maintain possession under pressure?
- How does pressure affect possession length?
- Does the team break pressure structurally?
- Which possessions create danger?
- Are dangerous possessions long or short?
- Does the team create chances through structure or transition?


In [17]:
temp_possessions = _possession_stats_base(match_df)
base_possessions = add_possession_opponent(temp_possessions, match_df)

time_control = _possession_stats_time(match_df)
progression = _possession_stats_progression(match_df)
pressure = _possession_stats_pressure(match_df)
outcome = _possession_stats_outcome(match_df)

merge_cols = ["match_id", "period", "possession_id", "possession_team"]
possession_summary = (
    base_possessions
    .merge(time_control, on=merge_cols, how="left")
    .merge(progression, on=merge_cols, how="left")
    .merge(pressure, on=merge_cols, how="left")
    .merge(outcome, on=merge_cols, how="left")
)

possession_summary.to_csv(PROCESSED_DIR / "possession_summary.csv", index=False)
dill.dump_session('saves/notebook_env.db')

In [18]:
dill.load_session('saves/notebook_env.db')

The progression metrics that I gathered, detailed in the columns are:

### Identity
- `match_id`: StatsBomb match id
- `period`: Match period: 1 first half, 2 second half, etc.
- `possession_id`: StatsBomb possession chain id within the match
- `possession_team`: Team that controls the possession chain
- `play_pattern`: How the possession started, e.g. Regular Play, From Throw In, From Corner
- `opponent`: The opposing team in the match

### Time / Control
- `start_minute`: Minute of the first event in the possession
- `start_second`: Second of the first event in the possession
- `end_minute`: Minute of the final event in the possession
- `end_second`: Second of the final event in the possession
- `start_time`: Possession start time in seconds from period clock, calculated as minute * 60 + second
- `end_time`: Possession end time in seconds from period clock, calculated as minute * 60 + second
- `clock_duration`: Elapsed clock time of the possession in seconds
- `action_duration`: Sum of StatsBomb event duration values within the possession
- `avg_event_duration`: Average event duration within the possession
- `event_count`: Total number of events in the possession chain
- `pass_count`: Number of Pass events by the possession team
- `carry_count`: Number of Carry events by the possession team
- `touch_count`: pass_count + carry_count; rough proxy for controlled ball actions

### Tempo / Deliberateness
- `events_per_second`: Event density, calculated as event_count divided by clock_duration
- `passes_per_second`: Pass tempo, calculated as pass_count divided by clock_duration
- `directness_ratio`: Net x progression divided by total pass/carry distance. Near 1 means direct forward play, near 0 means recycling, and negative means backward movement

### Progression
- `start_x`: X coordinate of the possession team’s first on-ball location
- `start_y`: Y coordinate of the possession team’s first on-ball location
- `end_x`: X coordinate of the possession team’s final on-ball/end location
- `end_y`: Y coordinate of the possession team’s final on-ball/end location
- `net_x_progression`: end_x - start_x; how far upfield the possession ended relative to where it began
- `total_x_progression`: Sum of all positive x movement from possession-team passes and carries
- `progressive_distance`: Sum of large positive x movements, currently movements with x gain >= 10
- `max_x_reached`: Furthest x coordinate reached during the possession
- `final_third_entry`: True if the possession reached x >= 80
- `box_entry`: True if the possession reached the box area, defined as x >= 102 and 18 <= y <= 62

### Pressure
- `pressure_event_count`: Number of explicit opponent Pressure events during the possession
- `under_pressure_count`: Number of possession-team events marked under_pressure
- `pressure_rate`: under_pressure_count divided by possession-team event count
- `pressured_pass_count`: Number of possession-team passes made under pressure
- `pressured_pass_completion`: Completion rate of possession-team passes made under pressure
- `counterpress_faced`: Number of opponent events marked counterpress during the possession

### Outcome / Danger
- `shot_count`: Number of shots by the possession team in the possession
- `xg_created`: Sum of shot_statsbomb_xg from possession-team shots
- `key_pass_count`: Number of shot assists or goal assists in the possession
- `turnover_type`: Label inferred from the final possession-team event, e.g. Pass - Incomplete, Shot - Saved, Out
- `ends_with_shot`: True if the final possession-team event was a shot
- `ends_in_final_third`: True if the possession team’s final location was in the final third
- `ends_in_box`: True if the possession team’s final location was in the box
